# Skin Cancer Detection System
## Google Colab Notebook
Presentation-ready notebook using HAM10000 dataset and CNN.

### Workflow Diagram
```
HAM10000 Dataset
      |
Metadata + Images
      |
Preprocessing
(BGR→RGB, Resize, Normalize)
      |
Label Encoding
      |
Train/Test Split
      |
Data Augmentation
      |
CNN Model
      |
Training
      |
Evaluation
      |
Prediction & Save Model
```

## 1. Install Libraries

In [ ]:
!pip install -q tensorflow opencv-python seaborn kaggle

## 2. Import Libraries

In [ ]:
import os,cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix,classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

## 3. Load Metadata

In [ ]:
df=pd.read_csv('HAM10000_metadata.csv')
df.head()

## 4. Map Image Paths

In [ ]:
paths={}
for d in ['HAM10000_images_part_1','HAM10000_images_part_2']:
    for f in os.listdir(d):
        paths[f.split('.')[0]]=os.path.join(d,f)
df['path']=df['image_id'].map(paths.get)

## 5. Disease Distribution

In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(x=df['dx'])
plt.show()

## 6. Image Preprocessing

In [ ]:
SIZE=128
images=[];labels=[]
for _,r in df.iterrows():
    img=cv2.imread(r['path'])
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    img=cv2.resize(img,(SIZE,SIZE))
    images.append(img)
    labels.append(r['dx'])
X=np.array(images,dtype=np.float32)/255.0
y=np.array(labels)

## 7. Label Encoding and Split

In [ ]:
enc=LabelEncoder()
y_enc=enc.fit_transform(y)
y_cat=to_categorical(y_enc)
X_train,X_test,y_train,y_test=train_test_split(X,y_cat,test_size=0.2,random_state=42,stratify=y_enc)

## 8. Data Augmentation

In [ ]:
datagen=ImageDataGenerator(rotation_range=30,zoom_range=0.2,horizontal_flip=True,vertical_flip=True,brightness_range=[0.8,1.2])

## 9. CNN Architecture

Input(128x128x3) → Conv32 → Pool → Conv64 → Pool → Conv128 → Pool → Flatten → Dense128 → Dropout → Softmax7

In [ ]:
model=Sequential([
Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
MaxPooling2D(),
Conv2D(64,(3,3),activation='relu'),
MaxPooling2D(),
Conv2D(128,(3,3),activation='relu'),
MaxPooling2D(),
Flatten(),
Dense(128,activation='relu'),
Dropout(0.5),
Dense(7,activation='softmax')
])
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

## 10. Train Model

In [ ]:
labels_train=np.argmax(y_train,axis=1)
cw=dict(enumerate(compute_class_weight(class_weight='balanced',classes=np.unique(labels_train),y=labels_train)))
early=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
history=model.fit(datagen.flow(X_train,y_train,batch_size=16),validation_data=(X_test,y_test),epochs=25,class_weight=cw,callbacks=[early])

## 11. Evaluate Model

In [ ]:
loss,acc=model.evaluate(X_test,y_test)
print(acc)
pred=np.argmax(model.predict(X_test),axis=1)
true=np.argmax(y_test,axis=1)
print(classification_report(true,pred,target_names=enc.classes_))
sns.heatmap(confusion_matrix(true,pred),annot=True,fmt='d',xticklabels=enc.classes_,yticklabels=enc.classes_)
plt.show()

## 12. Save Model

In [ ]:
model.save('skin_cancer_model.h5')